In [58]:
#RNA shenanigans
import RNA
import numpy as np
from Bio import SeqIO as io

#OS shenanigans
import subprocess
import os

#plotting shenanigans
import forgi.graph.bulge_graph as fgb
import forgi.visual.mplotlib as fvm
import matplotlib
matplotlib.use("Agg")

#tqdm because it's awesome
from tqdm import tqdm

#P.S. Python 3.11.14 with forgi 2.2.3 and viennarna 2.7.2 seems to work in terms of dependencies
#P.P.S We still have to figure out the issue with the rendering module not rendering certain sequences (see below)

In [59]:
def min_free_energy(seq):
    '''
    calculates a fold and minimum free energy given a sequence
    '''
    
    fc = RNA.fold_compound(seq) #Creating 2D fold
    (ss, mfe) = fc.mfe() #Calculating the minimum free energy for the fold
    
    return(mfe)

In [60]:
def pairing_probability(seq):
    '''
    calculates base pairing prospensity for each nucleotide given a sequence
    '''
    
    fc = RNA.fold_compound(seq) #Create 2D fold

    (propensity, ensemble_energy) = fc.pf() #Calculating propensity and ensemble energy for the next step
    
    bpp = np.asarray(fc.bpp()) #Calculating base pairing probabilities
    bpp = np.delete(np.delete(bpp, 0, 1), 0, 0) #Trimming first row and column

    bpp = bpp + bpp.transpose() #Making the matrix symmetric (i.e. the probability of base a bonding to b is the same as b bonding to a)

    assert(bpp.shape[0] == bpp.shape[1] and bpp.shape[0] == len(seq)) #Just as a little safety measure for now
    
    base_bpp = np.sum(bpp, 0)
    return(base_bpp)

In [61]:
def structure(seq):
    '''
    calculates a fold and outputs a dot-bracket notation given a sequence
    '''
    
    fc = RNA.fold_compound(seq) #Creating 2D fold
    (ss, mfe) = fc.mfe() #Calculating the minimum free energy for the fold
    return(ss)    

In [62]:
#Parsing the files Eren gave me

#Files: comparison.csv comparison_random.csv

lines = []
with open("comparison_random.csv", "r") as file:
    for i in file:
        lines.append(i)
head = lines[0]
lines = lines[1:]
real, ai = [], []
for i, line in enumerate(lines):
    seqs = line.split(",")
    real.append(seqs[0])
    ai.append(seqs[1].strip("\n"))
    #print(line.split(",")[0])

In [63]:
#Calculating the mfe and bpp of the sequences
mfe_output, bpp_output = [], []

for i in range(len(real)):
    mfe = [min_free_energy(real[i]), min_free_energy(ai[i])]
    bpp = [pairing_probability(real[i]), pairing_probability(ai[i])]
    #real_mfe = min_free_energy(real[i])
    #real_bpp = pairing_probability(real[i])
    #ai_mfe = min_free_energy(ai[i])
    #ai_bpp = pairing_probability(ai[i])
    mfe_output.append(mfe)
    bpp_output.append([ ['|'.join(map(str, pairing_probability(real[i])))],
                   ['|'.join(map(str, pairing_probability(ai[i])))]
                  ])

In [ ]:
#Writing sequences to files
with open("mfe_comparison_random_real_ai.csv", "w") as file:
    file.write(head)
    for line in mfe_output:
        file.write(str(line)+"\n")
with open("bpp_comparison_random_real_ai.csv", "w") as file:
    file.write(head)
    for line in bpp_output:
        file.write(str(line)+"\n")

In [64]:
cwd = os.getcwd()
if not os.path.exists(os.path.join(cwd, 'figs')):
    subprocess.run(["mkdir", "figs"])
os.chdir(os.path.join(cwd, 'figs'))

#plotting real data
for i, seq in tqdm(enumerate(real)):
    name = "real_RNA_" + str(i+1)
    bg = fgb.BulgeGraph.from_dotbracket(structure(seq), seq)

    fig, ax = plt.subplots(figsize=(40, 40))
    fvm.plot_rna(bg, ax=ax, text_kwargs={"fontweight":"black"}, lighten=0.7,
                 backbone_kwargs={"linewidth":1})
    matplotlib.pyplot.title(name, fontsize=50)
    matplotlib.pyplot.savefig(name)
    #plt.show()

#plotting ai data
for i, seq in tqdm(enumerate(ai)):
    name = "ai_RNA_" + str(i+1)
    bg = fgb.BulgeGraph.from_dotbracket(structure(seq), seq)

    fig, ax = plt.subplots(figsize=(40, 40))
    fvm.plot_rna(bg, ax=ax, text_kwargs={"fontweight":"black"}, lighten=0.7,
                 backbone_kwargs={"linewidth":1})
    matplotlib.pyplot.title(name, fontsize=50)
    print(seq)
    matplotlib.pyplot.savefig(name)
    #plt.show()

os.chdir("..")

#for future reference, there is a more elegant solution to this with:
#for name, j in [("real", real), ("ai", ai)]:
#    print(name, j[0])

0it [00:00, ?it/s]

real_RNA_1


/home/stepghin/anaconda3/envs/vienna_2/lib/python3.11/site-packages/forgi/visual/mplotlib.py:118: RuntimeWarning: invalid value encountered in divide
  norm_vec/=ftuv.magnitude(norm_vec)
1it [00:01,  1.66s/it]

real_RNA_2


2it [00:03,  1.70s/it]

...

1it [00:01,  1.68s/it]/home/stepghin/anaconda3/envs/vienna_2/lib/python3.11/site-packages/forgi/visual/mplotlib.py:372: RuntimeWarning: invalid value encountered in divide
  vec=vec/ftuv.magnitude(vec)
1it [00:02,  2.31s/it]


GTAAAACAAAAAAAAAATGGGCGGGAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAGAGGGGAAGAAAAAAAAAGGGGAAAAAGGAAAAGAGAAAAAGAAAAAAAAAAAAAAAAAAAGAAAGAGAGAAAAAAAAAAAAAGAGAGAAAGGAAAAAAAAAAAAAAAAAAAAGAAGGAAAAAAAAGAAAAAAAAAAAAAGGGGAGAAAAAAAAAATGATAAAAAAAAAAAAGAAGGAGAAAAAGAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAGGGGGGGAGGGAGGGGAGGAAAAAAAAAAAAAAAAAAGGAAGGAGAATAAAGGGAAAAGAAAAGATAAAAAAAAAAGAAGAGGAGGAAAAAAAAAAAAAAAGAGAGATGAAAAAAAAATTTATTTTTGATACACCTCCCTCTTTAGGGATGATTTTTTGGTCGTTTTTTTTATTAATAAAAAAAAAAAAGGGGGGCCCGCGGTCTTAAATTATAAGGGATGTTTTAAAAAAAAAAGGACGGGGAAGAAAAAAAAAATAAAAAAAACCAAAAAAAACGAAGAAATAGAGAAAAAAAAAAAAAATAAAAAAAGTTAAAAAAAAGAGGGGGGGGGGTTTTATTTATGAGTAAAGGGGTAAGTAATTTTTTTTTTTACAATCTTATTATTTTTTTGATTTTGTTGGTGCGGGCGCAAAAGAGGGTGATGTGGGGGGGGGAGGGGCCCCATTTTTTGGGGGGGAGGGGGGAGAATGTAGTTAAATTGAAG


StopIteration: 

Error in callback <function _draw_all_if_interactive at 0x7688cf542f20> (for post_execute), with arguments args (),kwargs {}:


StopIteration: 

In [15]:
#Issue with the 2nd and 3rd sequences for AI in from the comparison_random.csv file yield errors with the rendering module 
#Trying to reproduce the ai_RNA_2 issue from the random sequences
#When I tried rendering these with an online tool, the structure looked weird, and had an enormous loop
#Perhaps a 3D structure is needed

seq = "GTAAAACAAAAAAAAAATGGGCGGGAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAGAGGGGAAGAAAAAAAAAGGGGAAAAAGGAAAAGAGAAAAAGAAAAAAAAAAAAAAAAAAAGAAAGAGAGAAAAAAAAAAAAAGAGAGAAAGGAAAAAAAAAAAAAAAAAAAAGAAGGAAAAAAAAGAAAAAAAAAAAAAGGGGAGAAAAAAAAAATGATAAAAAAAAAAAAGAAGGAGAAAAAGAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAGGGGGGGAGGGAGGGGAGGAAAAAAAAAAAAAAAAAAGGAAGGAGAATAAAGGGAAAAGAAAAGATAAAAAAAAAAGAAGAGGAGGAAAAAAAAAAAAAAAGAGAGATGAAAAAAAAATTTATTTTTGATACACCTCCCTCTTTAGGGATGATTTTTTGGTCGTTTTTTTTATTAATAAAAAAAAAAAAGGGGGGCCCGCGGTCTTAAATTATAAGGGATGTTTTAAAAAAAAAAGGACGGGGAAGAAAAAAAAAATAAAAAAAACCAAAAAAAACGAAGAAATAGAGAAAAAAAAAAAAAATAAAAAAAGTTAAAAAAAAGAGGGGGGGGGGTTTTATTTATGAGTAAAGGGGTAAGTAATTTTTTTTTTTACAATCTTATTATTTTTTTGATTTTGTTGGTGCGGGCGCAAAAGAGGGTGATGTGGGGGGGGGAGGGGCCCCATTTTTTGGGGGGGAGGGGGGAGAATGTAGTTAAATTGAAG"

name = "ai_RNA_" + str(2)
bg = fgb.BulgeGraph.from_dotbracket(structure(seq), seq)

fig, ax = plt.subplots(figsize=(40, 40))
fvm.plot_rna(bg, ax=ax, text_kwargs={"fontweight":"black"}, lighten=0.7, backbone_kwargs={"linewidth":1})
plt.title("real RNA, fontsize=50")

plt.savefig("yes")
#plt.show()


StopIteration: 

Error in callback <function _draw_all_if_interactive at 0x7688cf542f20> (for post_execute), with arguments args (),kwargs {}:


StopIteration: 

StopIteration: 

<Figure size 4000x4000 with 1 Axes>